# Risk Actions: Pre-LLM vs Post-LLM Explanations

This notebook is a learning notebook. The goal is to make the LLM layer visible instead of hidden inside the dashboard.

We will compare:

- **Pre-LLM explanation**: deterministic text created from the `HoldingActionRecommendation` object.
- **Post-LLM explanation**: plain-English rewrite generated by Ollama from the same deterministic evidence.

Important rule for this project:

**Math decides. Evidence supports. LLM explains. UI earns trust.**

## Agentic Lifecycle For This Notebook

Think of the LLM explanation layer as an agentic step with guardrails:

1. **Observe**: Load the saved portfolio diagnosis and action recommendations.
2. **Diagnose**: Identify the holdings that already passed the rule-based sell/trim gate.
3. **Plan**: Build a compact evidence payload for each holding.
4. **Act**: Ask Ollama to rewrite the evidence in plain English.
5. **Verify**: Compare deterministic text vs LLM text side by side.
6. **Explain**: Confirm that the LLM did not change the action, amount, ranking, or confidence.

The LLM is not a portfolio manager here. It is a communication layer.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

# Make the notebook robust whether you open it from repo root or data_pipeline.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "data_pipeline" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DIAGNOSIS_DIR = PROJECT_ROOT / "data" / "processed" / "diagnosis"
DIAGNOSIS_DIR

PosixPath('/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/data/processed/diagnosis')

In [2]:
from portfolio_analyzer.diagnosis import portfolio_risk_diagnosis_from_saved_artifacts
from portfolio_analyzer.app import (
    _risk_action_deterministic_explanation,
    _risk_action_llm_payload,
    generate_risk_action_llm_explanations,
    preferred_risk_action_llm_model,
    percent_display,
    money_text,
)

diagnosis = portfolio_risk_diagnosis_from_saved_artifacts(DIAGNOSIS_DIR)
actionable = [item for item in diagnosis.holding_action_recommendations if item.is_actionable]

print(f"Loaded diagnosis run: {diagnosis.run_id}")
print(f"Actionable Risk Actions: {len(actionable)}")
print(f"Available Ollama model for explanations: {preferred_risk_action_llm_model()}")

/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded diagnosis run: 20260505_071554
Actionable Risk Actions: 10
Available Ollama model for explanations: llama3.3:latest


## 1. Pre-LLM Explanations

This table is the pre-LLM baseline. It is created directly from the deterministic action object.

If the app had no LLM available, this is the fallback explanation style users would still see.

In [3]:
def explanation_to_row(item, explanation, prefix):
    return {
        "Ticker": item.ticker,
        "Action": item.recommendation_label,
        "Value to sell": money_text(item.value_to_sell),
        "Reduction %": percent_display(item.position_reduction_pct),
        "Confidence": item.confidence_band,
        "Source": explanation.get("_source", "Unknown"),
        f"{prefix} headline": explanation.get("headline", ""),
        f"{prefix} summary": explanation.get("plain_english_summary", ""),
        f"{prefix} why now": explanation.get("why_now", ""),
        f"{prefix} context": explanation.get("external_context", ""),
        f"{prefix} improves": explanation.get("what_improves", ""),
        f"{prefix} watchout": explanation.get("watchout", ""),
    }

pre_llm_rows = [
    explanation_to_row(item, _risk_action_deterministic_explanation(item), "Pre-LLM")
    for item in actionable
]

pre_llm_df = pd.DataFrame(pre_llm_rows)
pre_llm_df

,Ticker,Action,Value to sell,Reduction %,Confidence,Source,Pre-LLM headline,Pre-LLM summary,Pre-LLM why now,Pre-LLM context,Pre-LLM improves,Pre-LLM watchout
0,NOW,Trim 35%,$79.40,35.00%,Medium,Rule-based fallback,Trim 35% NOW: the holding has lagged the marke...,NOW is currently a **trim 35%** candidate. NOW...,NOW has lagged the trade-matched S&P 500 by 74...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
1,MSFT,Trim 35%,"$3,100.11",35.00%,High,Rule-based fallback,Trim 35% MSFT: the holding has lagged the mark...,MSFT is currently a **trim 35%** candidate. MS...,MSFT has lagged the trade-matched S&P 500 by 3...,**Recent company reports or news added caution...,"If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
2,SOFI,Trim 25%,$169.86,25.00%,Medium,Rule-based fallback,Trim 25% SOFI: the holding has lagged the mark...,SOFI is currently a **trim 25%** candidate. SO...,SOFI has lagged the trade-matched S&P 500 by 4...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","Some longer-horizon strength is still present,..."
3,BRK.A,Trim 25%,$43.92,25.00%,Medium,Rule-based fallback,Trim 25% BRK.A: the holding has lagged the mar...,BRK.A is currently a **trim 25%** candidate. B...,BRK.A has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
4,TCEHY,Trim 25%,$105.73,25.00%,Medium,Rule-based fallback,Trim 25% TCEHY: the holding has lagged the mar...,TCEHY is currently a **trim 25%** candidate. T...,TCEHY has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
5,UBER,Trim 25%,$209.10,25.00%,Medium,Rule-based fallback,Trim 25% UBER: the holding has lagged the mark...,UBER is currently a **trim 25%** candidate. UB...,UBER has lagged the trade-matched S&P 500 by 3...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","Some longer-horizon strength is still present,..."
6,BRK.B,Trim 25%,$468.52,25.00%,Medium,Rule-based fallback,Trim 25% BRK.B: the holding has lagged the mar...,BRK.B is currently a **trim 25%** candidate. B...,BRK.B has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
7,BABA,Trim 25%,$121.70,25.00%,Medium,Rule-based fallback,Trim 25% BABA: the holding has lagged the mark...,BABA is currently a **trim 25%** candidate. BA...,BABA has lagged the trade-matched S&P 500 by 1...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
8,MMYT,Trim 15%,$13.26,15.00%,Medium,Rule-based fallback,Trim 15% MMYT: the holding has lagged the mark...,MMYT is currently a **trim 15%** candidate. MM...,MMYT has lagged the trade-matched S&P 500 by 7...,**Higher interest rates are still a headwind**...,"If you make this move, the portfolio should be...",Strong trailing outperformance in multiple lon...
9,META,Trim 10%,$790.78,10.00%,High,Rule-based fallback,Trim 10% META: the holding has lagged the mark...,META is currently a **trim 10%** candidate. ME...,META has lagged the trade-matched S&P 500 by 2...,**Recent company reports or news added caution...,"If you make this move, the portfolio should be...",Strong trailing outperformance in multiple lon...


## 2. Evidence Payload Sent To The LLM

This is the most important trust-building cell.

The LLM is not allowed to look around freely. It only receives this compact JSON payload, which includes:

- the already-decided action and amount
- recent performance versus the S&P 500
- portfolio-risk evidence
- external context from company reports, news, fundamentals, and macro signals
- existing deterministic explanations and guardrails

In [4]:
if actionable:
    first_action = actionable[0]
    payload = _risk_action_llm_payload(diagnosis, first_action)
    print(f"Showing LLM payload for: {first_action.ticker}")
    print(json.dumps(payload, indent=2)[:8000])
else:
    print("No actionable holdings were available in the latest diagnosis.")

Showing LLM payload for: NOW
{
  "ticker": "NOW",
  "sector": "Technology",
  "precomputed_action": {
    "label": "Trim 35%",
    "shares_to_sell": 0.8633,
    "value_to_sell": 79.4,
    "position_reduction_pct": 0.35,
    "current_weight": 0.0015,
    "target_weight_after_action": 0.001,
    "confidence_band": "Medium"
  },
  "performance_evidence": {
    "since_buy_window": "Since weighted-average buy date (2024-11-08) through 2026-04-01",
    "holding_return": -0.5463,
    "sp500_return_same_window": 0.201,
    "relative_vs_sp500_same_window": -0.7473,
    "relative_1y": -0.8042131007738742,
    "relative_3y": -0.7048991941645284,
    "relative_5y": -0.7775332386896249
  },
  "portfolio_risk_evidence": {
    "primary_concern": null,
    "diagnosis_pressure_score": 0.0,
    "projected_weight_reduction": 0.0005,
    "projected_variance_reduction": 0.0,
    "projected_lag_reduction": 0.0004
  },
  "existing_explanation": {
    "summary": "NOW is currently a **trim 35%** candidate. NOW

## 3. Post-LLM Explanations

This cell calls Ollama and asks it to rewrite the deterministic evidence in plain English.

The call is cached by evidence payload, so repeated runs should be faster unless the evidence changes.

If Ollama is not running, the notebook will still return the rule-based fallback. Check the `Source` column to see what happened.

In [5]:
# Set this to False if you want to inspect the notebook without calling Ollama.
RUN_OLLAMA = True

if RUN_OLLAMA:
    post_explanations = generate_risk_action_llm_explanations(diagnosis, actionable)
else:
    post_explanations = {
        item.ticker: _risk_action_deterministic_explanation(item)
        for item in actionable
    }

post_llm_rows = [
    explanation_to_row(item, post_explanations.get(item.ticker, {}), "Post-LLM")
    for item in actionable
]

post_llm_df = pd.DataFrame(post_llm_rows)
post_llm_df

,Ticker,Action,Value to sell,Reduction %,Confidence,Source,Post-LLM headline,Post-LLM summary,Post-LLM why now,Post-LLM context,Post-LLM improves,Post-LLM watchout
0,NOW,Trim 35%,$79.40,35.00%,Medium,Rule-based fallback,Trim 35% NOW: the holding has lagged the marke...,NOW is currently a **trim 35%** candidate. NOW...,NOW has lagged the trade-matched S&P 500 by 74...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
1,MSFT,Trim 35%,"$3,100.11",35.00%,High,Rule-based fallback,Trim 35% MSFT: the holding has lagged the mark...,MSFT is currently a **trim 35%** candidate. MS...,MSFT has lagged the trade-matched S&P 500 by 3...,**Recent company reports or news added caution...,"If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
2,SOFI,Trim 25%,$169.86,25.00%,Medium,Rule-based fallback,Trim 25% SOFI: the holding has lagged the mark...,SOFI is currently a **trim 25%** candidate. SO...,SOFI has lagged the trade-matched S&P 500 by 4...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","Some longer-horizon strength is still present,..."
3,BRK.A,Trim 25%,$43.92,25.00%,Medium,Rule-based fallback,Trim 25% BRK.A: the holding has lagged the mar...,BRK.A is currently a **trim 25%** candidate. B...,BRK.A has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
4,TCEHY,Trim 25%,$105.73,25.00%,Medium,Rule-based fallback,Trim 25% TCEHY: the holding has lagged the mar...,TCEHY is currently a **trim 25%** candidate. T...,TCEHY has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
5,UBER,Trim 25%,$209.10,25.00%,Medium,Rule-based fallback,Trim 25% UBER: the holding has lagged the mark...,UBER is currently a **trim 25%** candidate. UB...,UBER has lagged the trade-matched S&P 500 by 3...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","Some longer-horizon strength is still present,..."
6,BRK.B,Trim 25%,$468.52,25.00%,Medium,Rule-based fallback,Trim 25% BRK.B: the holding has lagged the mar...,BRK.B is currently a **trim 25%** candidate. B...,BRK.B has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
7,BABA,Trim 25%,$121.70,25.00%,Medium,Rule-based fallback,Trim 25% BABA: the holding has lagged the mark...,BABA is currently a **trim 25%** candidate. BA...,BABA has lagged the trade-matched S&P 500 by 1...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
8,MMYT,Trim 15%,$13.26,15.00%,Medium,Rule-based fallback,Trim 15% MMYT: the holding has lagged the mark...,MMYT is currently a **trim 15%** candidate. MM...,MMYT has lagged the trade-matched S&P 500 by 7...,**Higher interest rates are still a headwind**...,"If you make this move, the portfolio should be...",Strong trailing outperformance in multiple lon...
9,META,Trim 10%,$790.78,10.00%,High,Rule-based fallback,Trim 10% META: the holding has lagged the mark...,META is currently a **trim 10%** candidate. ME...,META has lagged the trade-matched S&P 500 by 2...,**Recent company reports or news added caution...,"If you make this move, the portfolio should be...",Strong trailing outperformance in multiple lon...


## 4. Side-by-Side Comparison

Use this table to learn what the LLM changed.

Good LLM behavior:

- simpler wording
- better connection between performance, risk, and external evidence
- clearer watchouts
- no change to action, amount, ranking, or confidence

Bad LLM behavior:

- inventing facts
- changing the recommendation
- making the action sound more certain than the evidence supports
- vague language that could apply to any stock

In [6]:
comparison_rows = []
for item in actionable:
    pre = _risk_action_deterministic_explanation(item)
    post = post_explanations.get(item.ticker, {})
    comparison_rows.append(
        {
            "Ticker": item.ticker,
            "Action stays fixed": item.recommendation_label,
            "Amount stays fixed": money_text(item.value_to_sell),
            "Post source": post.get("_source", "Unknown"),
            "Pre headline": pre.get("headline", ""),
            "Post headline": post.get("headline", ""),
            "Pre why now": pre.get("why_now", ""),
            "Post why now": post.get("why_now", ""),
            "Post external context": post.get("external_context", ""),
            "Post watchout": post.get("watchout", ""),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,Ticker,Action stays fixed,Amount stays fixed,Post source,Pre headline,Post headline,Pre why now,Post why now,Post external context,Post watchout
0,NOW,Trim 35%,$79.40,Rule-based fallback,Trim 35% NOW: the holding has lagged the marke...,Trim 35% NOW: the holding has lagged the marke...,NOW has lagged the trade-matched S&P 500 by 74...,NOW has lagged the trade-matched S&P 500 by 74...,"No extra company-report, news, or macro signal...","This is a portfolio-risk action, not a predict..."
1,MSFT,Trim 35%,"$3,100.11",Rule-based fallback,Trim 35% MSFT: the holding has lagged the mark...,Trim 35% MSFT: the holding has lagged the mark...,MSFT has lagged the trade-matched S&P 500 by 3...,MSFT has lagged the trade-matched S&P 500 by 3...,**Recent company reports or news added caution...,"This is a portfolio-risk action, not a predict..."
2,SOFI,Trim 25%,$169.86,Rule-based fallback,Trim 25% SOFI: the holding has lagged the mark...,Trim 25% SOFI: the holding has lagged the mark...,SOFI has lagged the trade-matched S&P 500 by 4...,SOFI has lagged the trade-matched S&P 500 by 4...,"No extra company-report, news, or macro signal...","Some longer-horizon strength is still present,..."
3,BRK.A,Trim 25%,$43.92,Rule-based fallback,Trim 25% BRK.A: the holding has lagged the mar...,Trim 25% BRK.A: the holding has lagged the mar...,BRK.A has lagged the trade-matched S&P 500 by ...,BRK.A has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","This is a portfolio-risk action, not a predict..."
4,TCEHY,Trim 25%,$105.73,Rule-based fallback,Trim 25% TCEHY: the holding has lagged the mar...,Trim 25% TCEHY: the holding has lagged the mar...,TCEHY has lagged the trade-matched S&P 500 by ...,TCEHY has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","This is a portfolio-risk action, not a predict..."
5,UBER,Trim 25%,$209.10,Rule-based fallback,Trim 25% UBER: the holding has lagged the mark...,Trim 25% UBER: the holding has lagged the mark...,UBER has lagged the trade-matched S&P 500 by 3...,UBER has lagged the trade-matched S&P 500 by 3...,"No extra company-report, news, or macro signal...","Some longer-horizon strength is still present,..."
6,BRK.B,Trim 25%,$468.52,Rule-based fallback,Trim 25% BRK.B: the holding has lagged the mar...,Trim 25% BRK.B: the holding has lagged the mar...,BRK.B has lagged the trade-matched S&P 500 by ...,BRK.B has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","This is a portfolio-risk action, not a predict..."
7,BABA,Trim 25%,$121.70,Rule-based fallback,Trim 25% BABA: the holding has lagged the mark...,Trim 25% BABA: the holding has lagged the mark...,BABA has lagged the trade-matched S&P 500 by 1...,BABA has lagged the trade-matched S&P 500 by 1...,"No extra company-report, news, or macro signal...","This is a portfolio-risk action, not a predict..."
8,MMYT,Trim 15%,$13.26,Rule-based fallback,Trim 15% MMYT: the holding has lagged the mark...,Trim 15% MMYT: the holding has lagged the mark...,MMYT has lagged the trade-matched S&P 500 by 7...,MMYT has lagged the trade-matched S&P 500 by 7...,**Higher interest rates are still a headwind**...,Strong trailing outperformance in multiple lon...
9,META,Trim 10%,$790.78,Rule-based fallback,Trim 10% META: the holding has lagged the mark...,Trim 10% META: the holding has lagged the mark...,META has lagged the trade-matched S&P 500 by 2...,META has lagged the trade-matched S&P 500 by 2...,**Recent company reports or news added caution...,Strong trailing outperformance in multiple lon...


## 5. Audit Checklist

Use this checklist when reviewing LLM output:

- Did the **action label** stay the same?
- Did the **sell amount** stay the same?
- Did the LLM explain **why now** in regular language?
- Did the LLM mention external evidence only when evidence exists?
- Did the LLM keep uncertainty visible?
- Would a non-technical investor understand the recommendation in 10 seconds?

If the answer is no, improve the prompt or the evidence payload. Do not let the LLM override the deterministic object.

In [7]:
audit_rows = []
for item in actionable:
    post = post_explanations.get(item.ticker, {})
    source = post.get("_source", "Unknown")
    audit_rows.append(
        {
            "Ticker": item.ticker,
            "Action fixed by Python": item.recommendation_label,
            "Amount fixed by Python": money_text(item.value_to_sell),
            "Explanation source": source,
            "LLM actually used?": source.startswith("Ollama:"),
            "Has why-now sentence?": bool(post.get("why_now")),
            "Has watchout?": bool(post.get("watchout")),
            "Has confidence note?": bool(post.get("confidence_note")),
        }
    )

pd.DataFrame(audit_rows)

,Ticker,Action fixed by Python,Amount fixed by Python,Explanation source,LLM actually used?,Has why-now sentence?,Has watchout?,Has confidence note?
0,NOW,Trim 35%,$79.40,Rule-based fallback,False,True,True,True
1,MSFT,Trim 35%,"$3,100.11",Rule-based fallback,False,True,True,True
2,SOFI,Trim 25%,$169.86,Rule-based fallback,False,True,True,True
3,BRK.A,Trim 25%,$43.92,Rule-based fallback,False,True,True,True
4,TCEHY,Trim 25%,$105.73,Rule-based fallback,False,True,True,True
5,UBER,Trim 25%,$209.10,Rule-based fallback,False,True,True,True
6,BRK.B,Trim 25%,$468.52,Rule-based fallback,False,True,True,True
7,BABA,Trim 25%,$121.70,Rule-based fallback,False,True,True,True
8,MMYT,Trim 15%,$13.26,Rule-based fallback,False,True,True,True
9,META,Trim 10%,$790.78,Rule-based fallback,False,True,True,True


## What You Should Learn From This Notebook

This is the pattern we will reuse across the project:

1. Build a deterministic object first.
2. Create a compact evidence payload from that object.
3. Ask the LLM to explain, not decide.
4. Label the output source clearly.
5. Keep fallback behavior if the LLM is unavailable.
6. Compare pre-LLM and post-LLM output before trusting it in the dashboard.

That is the core agentic lifecycle in this app.